In [ ]:
# Process .mat file and convert to .nc

In [ ]:
import os
import h5py
import numpy as np
import xarray as xr

In [ ]:
# === Path config ===
DIR = "/glade/work/awells/air_quality/BMR/masks/region/"
in_file = "GBD_Region_Masks_0.10.mat"
file_path = os.path.join(DIR, in_file)

with h5py.File(file_path, "r") as f:
    # Dereference mask objects
    mask_refs = f["sGBDRegions/Mask"][0]
    name_refs = f["sGBDRegions/name"][0]

    lat = f["gLAT"][0]
    lon = f["gLON"][0]

    masks = []
    names = []
    regions = []

    for i, ref in enumerate(mask_refs):
        # Dereference mask
        mask = f[ref][()]
        masks.append(mask)

        # Region name
        name_ascii = f[name_refs[i]][()]
        name_str = "".join([chr(c[0]) for c in name_ascii])
        names.append(name_str)

# Confirm all masks have same shape before stacking
first_shape = masks[0].shape
if all(mask.shape == first_shape for mask in masks):
    # Stack into 3D array: (country, lat, lon)
    mask_array = np.stack(masks)
    data_array = xr.DataArray(
        mask_array,
        dims=["region", "lat", "lon"],
        coords={
            "region": names,
            "lat": lat,
            "lon": lon,
        },
        name="region_mask"
    )
else:
    raise ValueError("Masks have varying shapes – cannot stack into DataArray.")

description = ("DataArray that contains 0.1ºx0.1º gridded masks for all 21"
               "world regions")
cite = ("McDuffie, E., Brauer, M., Martin, R., Spadaro, J., Burnett, R., "
        "Hammer, M., & van Donkelaar, A. (2021). GBD-MAPS-Global: "
        "Analysis Input Dataset [Data set]. Zenodo. "
        "https://doi.org/10.5281/zenodo.4642700")

data_array.attrs["description"] = description
data_array.attrs["citation"] = cite

out_file = "GBD_Region_Masks_0.10.nc"
out_path = os.path.join(DIR, out_file)
data_array.to_netcdf(out_path)

print("All processing complete.")